# Collaborative Filtering — Movie Recommendation System

> **Dataset:** MovieLens 100K &nbsp;|&nbsp; **Techniques:** User-User CF · Item-Item CF · Cosine · Pearson · Adjusted Cosine  
> **Goal:** Predict how a user will rate an unseen movie by leveraging ratings from similar users or similar items.

---

## 📑 Table of Contents

1. [Imports & Setup](#1-imports--setup)  
2. [Data Loading](#2-data-loading)  
3. [User-Item Rating Matrix](#3-user-item-rating-matrix)  
4. [Similarity Metrics](#4-similarity-metrics)  
5. [User-User Collaborative Filtering](#5-user-user-collaborative-filtering)  
6. [Item-Item Collaborative Filtering](#6-item-item-collaborative-filtering)  
7. [Generating Top-N Recommendations](#7-generating-top-n-recommendations)  
8. [Evaluation — Precision@K](#8-evaluation--precisionk)

---

## Background

**Collaborative Filtering (CF)** is one of the most widely deployed recommendation paradigms.  
The core assumption: *users who agreed in the past will agree in the future*.

There are two main memory-based flavours:

| Approach | Idea | Best when |
|---|---|---|
| **User-User CF** | Find users similar to the target user; borrow their ratings | Many users, sparse items |
| **Item-Item CF** | Find items similar to the target item; use user's own past ratings | Many items, user preferences are stable |

Both approaches covered in this notebook use the **MovieLens 100K** dataset (943 users, 1,682 movies, 100,000 ratings).


## 1. Imports & Setup

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import time

## 2. Data Loading

The MovieLens 100K dataset ships as pipe- or tab-delimited plain-text files.  
We load three tables:

| File | Contents |
|---|---|
| `u.data` | (user_id, item_id, rating, timestamp) — 100,000 rows |
| `u.item` | Movie metadata including 19 binary genre flags |
| `u.user` | User demographics (age, gender, occupation, zip) |


In [3]:
# ── Dataset path (adjust if running locally) ──────────────────────────────────
basepath = "/kaggle/input/datasets/trishna8/movielens-100k-dataset/ml-100k/"

movies_cols = [
    'movie id', 'movie title', 'release date', 'video release date', 'IMDb URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
    'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery',
    'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]
users_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']

# ── Load data ──────────────────────────────────────────────────────────────────
ratings = pd.read_csv(
    f"{basepath}u.data", sep='\t', encoding="latin-1",
    header=None, names=['user_id', 'item_id', 'rating', 'timestamp']
)
movies = pd.read_csv(
    f"{basepath}u.item", sep="|", encoding="latin-1",
    header=None, names=movies_cols
)
users = pd.read_csv(f"{basepath}u.user", names=users_cols, sep='|')

print(f"Ratings : {ratings.shape[0]:,} rows × {ratings.shape[1]} cols")
print(f"Movies  : {movies.shape[0]:,} rows × {movies.shape[1]} cols")
print(f"Users   : {users.shape[0]:,} rows × {users.shape[1]} cols")


Ratings : 100,000 rows × 4 cols
Movies  : 1,682 rows × 24 cols
Users   : 943 rows × 5 cols


In [4]:
# Quick peek at the ratings table
ratings.head()


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


## 3. User-Item Rating Matrix

Collaborative filtering operates on a **user × item rating matrix** $R$ where  
$R_{ui}$ is the rating user $u$ gave item $i$, and `NaN` means unobserved.

> **Sparsity** is the fraction of entries that *are* rated.  
> MovieLens 100K is typically ~6% dense — the bulk of entries are missing,  
> which is exactly why we need smart similarity-based methods.


In [5]:
# Pivot ratings into a (users × items) matrix
ratings_pivot = pd.pivot_table(
    ratings, index='user_id', columns='item_id', values='rating'
)

rows, cols = ratings_pivot.shape
total_nan = ratings_pivot.isna().values.sum()
density = 1 - (total_nan / (rows * cols))

print(f"Matrix shape : {rows} users × {cols} items")
print(f"Total NaN    : {total_nan:,}")
print(f"Density      : {density:.2%}  (sparsity = {1-density:.2%})")


Matrix shape : 943 users × 1682 items
Total NaN    : 1,486,126
Density      : 6.30%  (sparsity = 93.70%)


In [31]:
# Example: rating vector for user 1 (NaN = not yet rated)
ratings_pivot


item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
940,NaN,NaN,NaN,2.0,NaN,NaN,4.0,5.0,3.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
941,5.0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Similarity Metrics

Choosing the right similarity measure is critical. Three are implemented here:

### 4a. Cosine Similarity
$$\text{cos}(u,v) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$

- Fills NaNs with 0 before computing (treats missing = didn't rate = 0).  
- Fast but ignores rating scale differences between users.

### 4b. Pearson Correlation
$$\text{pearson}(u,v) = \frac{\sum_{i \in I_{uv}}(r_{ui}-\bar{r}_u)(r_{vi}-\bar{r}_v)}{\|r_u - \bar{r}_u\| \|r_v - \bar{r}_v\|}$$

- Only uses **co-rated items** $I_{uv}$.  
- Mean-centers each user's ratings → accounts for generous vs. harsh raters.  
- `min_overlap=3` guards against spurious high similarity on a single co-rating.

### 4c. Adjusted Cosine (for Item-Item CF)
$$\text{adj\_cos}(i,j) = \frac{\sum_{u \in U_{ij}}(r_{ui} - \bar{r}_u)(r_{uj} - \bar{r}_u)}{\sqrt{\sum_{u}(r_{ui}-\bar{r}_u)^2} \sqrt{\sum_u(r_{uj}-\bar{r}_u)^2}}$$

- Subtracts **user mean** before computing cosine → corrects for user rating bias.  
- Standard choice for Item-Item CF.


In [6]:
def cosine_sim(x, y):
    """
    Cosine similarity between two rating vectors.
    NaN entries are treated as 0 (neutral) before computation.
    """
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    # Replace NaN with 0 so unrated items contribute nothing
    x_filled, y_filled = np.nan_to_num(x), np.nan_to_num(y)
    denom = np.linalg.norm(x_filled) * np.linalg.norm(y_filled)
    if denom == 0:
        return 0.0
    return float(np.dot(x_filled, y_filled) / denom)


def pearson_sim(x, y, min_overlap=3):
    """
    Pearson correlation over co-rated items only.
    Returns 0 if fewer than min_overlap items are co-rated.
    """
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    # Mask: keep only positions where BOTH users rated
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_common, y_common = x[mask], y[mask]

    if len(x_common) < min_overlap:
        return 0.0  # Not enough co-ratings → unreliable

    # Mean-center within the co-rated subset
    x_centered = x_common - x_common.mean()
    y_centered = y_common - y_common.mean()

    denom = np.linalg.norm(x_centered) * np.linalg.norm(y_centered)
    if denom == 0:
        return 0.0
    return float(np.dot(x_centered, y_centered) / denom)


def adjusted_cosine_sim(x, y):
    """
    Adjusted cosine: cosine on co-rated items.
    Used for item-item similarity; caller should pre-subtract user means.
    """
    x, y = np.array(x, dtype=float), np.array(y, dtype=float)
    mask = ~np.isnan(x) & ~np.isnan(y)
    x_common, y_common = x[mask], y[mask]

    denom = np.linalg.norm(x_common) * np.linalg.norm(y_common)
    if denom == 0:
        return 0.0
    return float(np.dot(x_common, y_common) / denom)


def compute_similarity(x, y, metric):
    """Dispatch to the requested similarity function."""
    if metric == 'cosine':
        return cosine_sim(x, y)
    elif metric == 'pearson':
        return pearson_sim(x, y)
    elif metric == 'adjusted_cosine':
        return adjusted_cosine_sim(x, y)
    else:
        raise ValueError(f"Unknown metric '{metric}'. Choose: cosine | pearson | adjusted_cosine")


#  Sanity check
x_test = [3, 2, np.nan, 2]
y_test = [2, np.nan, 3, 1]
print(f"Cosine sim           : {cosine_sim(x_test, y_test):.4f}")
print(f"Pearson sim          : {pearson_sim(x_test, y_test):.4f}")
print(f"Adjusted cosine sim  : {adjusted_cosine_sim(x_test, y_test):.4f}")


Cosine sim           : 0.5186
Pearson sim          : 0.0000
Adjusted cosine sim  : 0.9923


## 5. User-User Collaborative Filtering

### How it works

1. **Find neighbours**: for target user $u$ and target item $i$, collect all users who rated $i$.  
2. **Rank by similarity**: compute similarity(u, each neighbour) and keep top-$k$.  
3. **Predict**: aggregate neighbour ratings, weighted by their similarity to $u$.

Two prediction variants are implemented:

#### Weighted Average (plain)
$$\hat{r}_{ui} = \frac{\sum_{v \in N(u)} \text{sim}(u,v) \cdot r_{vi}}{\sum_{v \in N(u)} |\text{sim}(u,v)|}$$

#### Mean-Centered (bias-corrected) ✅ preferred
$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N(u)} \text{sim}(u,v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in N(u)} |\text{sim}(u,v)|}$$

The mean-centered version adjusts for the fact that some users rate everything highly while others are harsher critics.


In [7]:
def predict_mean_centered_rating(user_id, item_id, sim_metric, df_ratings, topn=10):
    """
    Predict rating for (user_id, item_id) using mean-centered User-User CF.

    Steps:
      1. Filter neighbours who have rated item_id.
      2. Compute similarity of each neighbour to the target user.
      3. Predict using bias-corrected weighted average.

    Parameters
    ----------
    user_id    : int   — target user
    item_id    : int   — target item
    sim_metric : str   — 'cosine' | 'pearson' | 'adjusted_cosine'
    df_ratings : DataFrame — user-item pivot (rows=users, cols=items)
    topn       : int   — neighbourhood size

    Returns
    -------
    float — predicted rating
    """
    target_ratings = df_ratings.loc[user_id].values
    target_mean    = df_ratings.loc[user_id].mean()
    rating_cols    = df_ratings.columns

    # Keep only neighbours who have actually rated item_id
    df_neighbours = df_ratings.dropna(subset=[item_id]).copy()

    # Compute similarity of every neighbour to the target user
    df_neighbours['similarity'] = df_neighbours[rating_cols].apply(
        lambda row: compute_similarity(target_ratings, row.values, sim_metric), axis=1
    )
    # Compute each neighbour's mean rating (for bias correction)
    df_neighbours['mean'] = df_neighbours[rating_cols].mean(axis=1)

    # Exclude the target user themselves
    df_neighbours = df_neighbours.drop(index=user_id, errors='ignore')

    # Select top-N most similar neighbours
    top_n = df_neighbours.sort_values('similarity', ascending=False).head(topn)

    numerator, denominator = 0.0, 0.0
    for v in top_n.index:
        sim = top_n.loc[v, 'similarity']
        r_vi = top_n.loc[v, item_id]
        r_mean = top_n.loc[v, 'mean']
        numerator += sim * (r_vi - r_mean)   # bias-corrected contribution
        denominator += abs(sim)

    if denominator == 0:
        return target_mean   # fallback: user's own mean

    return target_mean + (numerator / denominator)


def predict_user_rating(user_id, item_id, sim_metric, df_ratings, topn=10):
    """
    Predict rating for (user_id, item_id) using plain weighted-average User-User CF.

    Parameters
    ----------
    Same as predict_mean_centered_rating.

    Returns
    -------
    float — predicted rating
    """
    target_ratings = df_ratings.loc[user_id].values
    target_mean    = df_ratings.loc[user_id].mean()
    rating_cols    = df_ratings.columns

    # Keep neighbours who have rated item_id, then exclude target user
    df_neighbours = df_ratings.dropna(subset=[item_id]).copy()
    df_neighbours = df_neighbours.drop(index=user_id, errors='ignore')

    # Compute pairwise similarities
    df_neighbours['similarity'] = df_neighbours[rating_cols].apply(
        lambda row: compute_similarity(target_ratings, row.values, sim_metric), axis=1
    )

    # Select top-N neighbours
    top_n = df_neighbours.sort_values('similarity', ascending=False).head(topn)

    numerator, denominator = 0.0, 0.0
    for v in top_n.index:
        sim  = top_n.loc[v, 'similarity']
        r_vi = top_n.loc[v, item_id]
        numerator   += sim * r_vi
        denominator += abs(sim)

    if denominator == 0:
        return target_mean   # fallback

    return numerator / denominator


In [34]:
# Demo predictions 
# Predict how user 2 would rate movie 5

pred_mc = predict_mean_centered_rating(2, 5, 'pearson', ratings_pivot)
pred_wa = predict_user_rating(2, 5, 'cosine', ratings_pivot)

print(f"Mean-centered (Pearson) prediction for user=2, movie=5 : {pred_mc:.3f}")
print(f"Weighted avg  (Cosine)  prediction for user=2, movie=5 : {pred_wa:.3f}")
print(f"Actual rating scale: 1–5")


Mean-centered (Pearson) prediction for user=2, movie=5 : 3.530
Weighted avg  (Cosine)  prediction for user=2, movie=5 : 3.398
Actual rating scale: 1–5


## 6. Item-Item Collaborative Filtering

### How it works

Instead of finding *similar users*, we find *similar items* to the target item $i$:

1. **Mean-center** the rating matrix by subtracting each user's mean rating (removes user bias).  
2. **Compute adjusted cosine similarity** between the target item and all other items rated by user $u$.  
3. **Predict** using weighted average of the user's ratings on those similar items.

$$\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{j \in S(i,u)} \text{sim}(i,j) \cdot (r_{uj} - \bar{r}_u)}{\sum_{j \in S(i,u)} |\text{sim}(i,j)|}$$

### Why Item-Item is often preferred in production

- Item similarity is more **stable** over time than user similarity.  
- Once item-item similarities are pre-computed, serving recommendations is $O(k)$.  
- Scales better when #items $\ll$ #users.


In [8]:
def predict_rating_item_item(user_id, item_id, metric, df_ratings, topn=100):
    """
    Predict rating for (user_id, item_id) using Item-Item CF with adjusted cosine.

    Steps:
      1. Build a mean-subtracted matrix (user bias removed).
      2. For each other item, compute similarity to target item using co-rated users.
      3. Weighted-average the user's ratings on top-k similar items.

    Parameters
    ----------
    user_id    : int   — target user
    item_id    : int   — target item (the item we want to predict for)
    metric     : str   — similarity metric (typically 'adjusted_cosine')
    df_ratings : DataFrame — user-item pivot
    topn       : int   — max neighbours to consider

    Returns
    -------
    float — predicted rating
    """
    df = df_ratings.copy()

    # Compute per-user mean ratings (row-wise)
    user_means = df.mean(axis=1)

    # Subtract user mean from every rating → remove user-level bias
    df_centered = df.sub(user_means, axis=0)

    item_similarities = []

    for col in df_centered.columns:
        if col == item_id:
            continue  # Skip the target item itself

        x = df_centered[col].values
        y = df_centered[item_id].values

        # Only compute if at least 3 users co-rated both items
        mask    = ~np.isnan(x) & ~np.isnan(y)
        overlap = mask.sum()
        if overlap > 2:
            sim = compute_similarity(x, y, metric)
            item_similarities.append({'Item': col, 'Similarity': sim, 'Overlap': overlap})

    if not item_similarities:
        # No neighbours found → fall back to user's mean rating
        return float(user_means.loc[user_id])

    item_sim_df = pd.DataFrame(item_similarities)

    # Sort by similarity (then overlap as tiebreaker), keep top-N
    item_sim_topn = (
        item_sim_df
        .sort_values(['Similarity', 'Overlap'], ascending=False)
        .head(topn)
        .set_index('Item')
    )

    numerator, denominator = 0.0, 0.0

    for item_j in item_sim_topn.index:
        # Use mean-centered rating that user gave to item_j
        r_uj_centered = df_centered.loc[user_id, item_j]

        if pd.isna(r_uj_centered):
            continue  # User hasn't rated this neighbour → skip

        sim = item_sim_topn.loc[item_j, 'Similarity']
        numerator   += sim * r_uj_centered
        denominator += abs(sim)

    if denominator == 0:
        return float(user_means.loc[user_id])  # fallback

    # Un-center: add back target user's mean
    return float(user_means.loc[user_id] + (numerator / denominator))


In [9]:
# ── Demo ──────────────────────────────────────────────────────────────────────
pred_ii = predict_rating_item_item(5, 2, 'adjusted_cosine', ratings_pivot)
print(f"Item-Item (Adjusted Cosine) prediction for user=5, movie=2 : {pred_ii:.3f}")


Item-Item (Adjusted Cosine) prediction for user=5, movie=2 : 1.726


## 7. Generating Top-N Recommendations

To recommend movies to a user, we:

1. Identify all movies the user **has not yet rated**.
2. Predict a rating for each unrated movie using the CF model.
3. Sort by predicted rating and return the top-$N$.

> This is $O(|items|)$ predictions per user — very slow for large datasets.  
> In production, candidate generation + ANN lookup (FAISS, ScaNN) replace this.


In [10]:
def get_top_n_movies_for_user(df_ratings, topn, user_id, metric, prediction_func):
    """
    Generate top-N movie recommendations for a user.

    Parameters
    ----------
    df_ratings      : DataFrame — user-item pivot (rows=users, cols=items)
    topn            : int       — number of recommendations to return
    user_id         : int       — target user
    metric          : str       — similarity metric for prediction_func
    prediction_func : callable  — one of predict_user_rating,
                                   predict_mean_centered_rating,
                                   predict_rating_item_item

    Returns
    -------
    DataFrame with columns ['Item', 'Prediction'], sorted descending.
    """
    unrated_preds = []

    for item_id in df_ratings.columns:
        # Skip items the user has already rated
        if not pd.isna(df_ratings.loc[user_id, item_id]):
            continue
        pred = prediction_func(user_id, item_id, metric, df_ratings)
        unrated_preds.append({'Item': item_id, 'Prediction': pred})

    df_preds = pd.DataFrame(unrated_preds)
    return df_preds.sort_values('Prediction', ascending=False).head(topn)


In [38]:
#  Example recommendations 
start = time.time()
print("=== Item-Item CF (Adjusted Cosine) — Top 5 for User 2 ===")
ii_recs = get_top_n_movies_for_user(
    ratings_pivot, 5, user_id=2,
    metric='adjusted_cosine', prediction_func=predict_rating_item_item
)
end = time.time()
print(f"Time taken is {end-start:.2f} secs")
print(ii_recs.to_string(index=False))

print("\n=== User-User CF (Cosine, mean-centered) — Top 5 for User 5 ===")
start = time.time()
uu_recs = get_top_n_movies_for_user(
    ratings_pivot, 5, user_id=5,
    metric='cosine', prediction_func=predict_mean_centered_rating
)
end = time.time()
print(f"Time taken is {end-start:.2f} secs")
print(uu_recs.to_string(index=False))


=== Item-Item CF (Adjusted Cosine) — Top 5 for User 2 ===
Time taken is 240.14 secs
 Item  Prediction
 1389    6.419355
   41    5.000000
 1140    5.000000
  631    5.000000
  641    5.000000

=== User-User CF (Cosine, mean-centered) — Top 5 for User 5 ===
Time taken is 21.20 secs
 Item  Prediction
  814    4.776801
 1463    4.735849
 1536    4.578912
 1500    4.464718
 1467    4.379709


## Implementing cache

In [11]:
def build_cf_user_similarity_cache(df_ratings, metric=pearson_sim):
    """
    Pre-compute the full user × user similarity matrix.

    Parameters
    ----------
    df_ratings : DataFrame — user-item pivot (rows=users, cols=items)
    metric     : callable  — pairwise similarity function (default: pearson_sim)

    Returns
    -------
    DataFrame — symmetric (n_users × n_users) similarity matrix
    """
    users   = df_ratings.index.tolist()
    n_users = len(users)
    R       = df_ratings.to_numpy()

    sim_matrix = np.zeros((n_users, n_users), dtype=float)
    for i in range(n_users):
        sim_matrix[i, i] = 1.0
        for j in range(i):
            sim = metric(R[i], R[j])
            sim_matrix[i, j] = sim_matrix[j, i] = sim

    return pd.DataFrame(sim_matrix, index=users, columns=users)

user_sim_cache = build_cf_user_similarity_cache(ratings_pivot)
user_sim_cache

,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
1,1.000000,0.160841,0.112780,0.500000,0.420809,0.287159,0.258137,0.692086,-0.102062,-0.092344,...,0.061695,-0.260242,0.383733,0.029000,0.326744,0.534390,0.263289,0.205616,-0.180784,0.067549
2,0.160841,1.000000,0.067420,0.148522,0.327327,0.446269,0.643675,0.585491,0.242536,0.668145,...,0.021007,-0.271163,0.214017,0.561645,0.331587,0.000000,-0.011682,-0.062017,0.085960,0.479702
3,0.112780,0.067420,1.000000,-0.262600,0.000000,-0.109109,0.064803,0.291937,0.000000,0.311086,...,0.000000,0.000000,-0.045162,0.000000,-0.137523,0.000000,-0.104678,1.000000,-0.011792,0.000000
4,0.500000,0.148522,-0.262600,1.000000,0.000000,-0.581318,-0.266632,0.642938,0.000000,-0.301511,...,0.500000,0.000000,-0.203653,0.000000,0.375000,0.000000,0.850992,1.000000,0.412568,0.000000
5,0.420809,0.327327,0.000000,0.000000,1.000000,0.241817,0.175630,0.537400,0.577350,0.087343,...,0.229532,-0.500000,0.439286,0.608581,0.484211,0.880705,0.027038,0.468521,0.318163,0.346234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.534390,0.000000,0.000000,0.000000,0.880705,0.206315,0.142404,-0.333333,0.000000,0.316228,...,0.374351,-0.033059,0.471172,-0.275839,-0.073374,1.000000,-0.534522,-0.131306,-0.500000,-0.187317
940,0.263289,-0.011682,-0.104678,0.850992,0.027038,-0.024419,0.000931,0.320487,0.171499,0.158976,...,-0.125059,0.435286,-0.338327,-0.148608,0.110022,-0.534522,1.000000,0.632746,-0.022813,0.332497
941,0.205616,-0.062017,1.000000,1.000000,0.468521,0.399186,0.000000,0.166667,1.000000,0.420084,...,-0.500000,0.000000,0.273060,0.392953,-0.214147,-0.131306,0.632746,1.000000,-0.577350,-0.395285
942,-0.180784,0.085960,-0.011792,0.412568,0.318163,0.092349,0.452075,0.201328,0.707107,0.408994,...,0.438252,-0.870388,-0.216119,0.447214,0.244989,-0.500000,-0.022813,-0.577350,1.000000,0.277433


In [12]:
def build_cf_item_similarity_cache(df_ratings, metric=adjusted_cosine_sim):
    """
    Pre-compute the full item × item similarity matrix.

    Ratings are mean-centered by user before the similarity is computed
    so that ``adjusted_cosine_sim`` receives pre-subtracted vectors,
    as required by its contract.

    Parameters
    ----------
    df_ratings : DataFrame — user-item pivot (rows=users, cols=items)
    metric     : callable  — pairwise similarity function
                             (default: adjusted_cosine_sim)

    Returns
    -------
    DataFrame — symmetric (n_items × n_items) similarity matrix
    """
    items   = df_ratings.columns.tolist()
    n_items = len(items)

    # Subtract each user's mean rating to remove user-level bias
    user_means  = df_ratings.mean(axis=1)
    df_centered = df_ratings.sub(user_means, axis=0)
    R           = df_centered.to_numpy()

    sim_matrix = np.zeros((n_items, n_items), dtype=float)
    for i in range(n_items):
        sim_matrix[i, i] = 1.0
        for j in range(i):
            sim = metric(R[:, i], R[:, j])
            sim_matrix[i, j] = sim_matrix[j, i] = sim

    return pd.DataFrame(sim_matrix, index=items, columns=items)

item_sim_cache = build_cf_item_similarity_cache(ratings_pivot)
item_sim_cache

,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
1,1.000000,-0.138378,-0.168012,-0.084358,0.044092,0.230893,0.111053,0.266410,0.064981,0.015624,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-1.0
2,-0.138378,1.000000,0.106923,0.055509,0.045516,-0.125510,-0.139482,-0.023878,-0.347896,-0.113399,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
3,-0.168012,0.106923,1.000000,-0.387420,0.090021,0.448830,-0.226423,-0.497486,-0.196040,-0.405941,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,-0.084358,0.055509,-0.387420,1.000000,-0.301850,-0.206525,0.009621,0.210378,0.066502,0.051887,...,0.0,0.0,-1.0,-1.0,1.0,0.0,0.0,0.0,1.0,-1.0
5,0.044092,0.045516,0.090021,-0.301850,1.000000,-0.951402,-0.116917,0.039633,-0.222521,-0.541469,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
1679,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
1680,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
1681,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [13]:
# Prediction helpers 

def _predict_uu_cached(user_id, item_id, df_ratings, user_sim_cache,
                       user_means, topn_neighbors):
    """
    Mean-centered User-User prediction using a pre-computed similarity cache.

    Steps:
      1. Collect users who have rated item_id (excluding target user).
      2. Look up their pre-computed similarities from the cache.
      3. Keep top-k neighbours; return bias-corrected weighted average.
    """
    target_mean = user_means.loc[user_id]

    neighbours = df_ratings.index[df_ratings[item_id].notna()].difference([user_id])
    if neighbours.empty:
        return target_mean

    sims        = user_sim_cache.loc[user_id, neighbours]
    top_idx     = sims.nlargest(topn_neighbors).index
    sims_top    = sims.loc[top_idx]
    ratings_top = df_ratings.loc[top_idx, item_id]
    means_top   = user_means.loc[top_idx]

    num = (sims_top * (ratings_top - means_top)).sum()
    den = sims_top.abs().sum()
    return target_mean if den == 0.0 else (target_mean + (num / den))


def _predict_ii_cached(user_id, item_id, df_ratings, item_sim_cache,
                       user_means, topn_neighbors):
    """
    Mean-centered Item-Item prediction using a pre-computed similarity cache.

    Steps:
      1. Collect items the user HAS rated (candidate neighbours of item_id).
      2. Look up their pre-computed similarities from the cache.
      3. Keep top-k neighbours; return bias-corrected weighted average.
    """
    target_mean = user_means.loc[user_id]

    user_row       = df_ratings.loc[user_id]
    rated_by_user  = user_row.dropna().index.difference([item_id])

    if rated_by_user.empty or item_id not in item_sim_cache.index:
        return target_mean

    sims        = item_sim_cache.loc[item_id, rated_by_user]
    top_idx     = sims.nlargest(topn_neighbors).index
    sims_top    = sims.loc[top_idx]
    ratings_top = user_row.loc[top_idx]

    num = (sims_top * (ratings_top - target_mean)).sum()
    den = sims_top.abs().sum()
    return target_mean if den == 0.0 else (target_mean + (num / den))


# Main function

def predict_topn_cache(
    user_id,
    df_ratings,
    topn_items,
    mode='user',
    user_sim_cache=None,
    item_sim_cache=None,
    topn_neighbors=15,
):
    """
    Predict top-N movie recommendations using pre-computed similarity caches.

    Two modes are supported:

    * 'user' — Mean-centered **User-User CF**.
      For each unrated item, finds the top-k most similar users who rated it
      and returns a bias-corrected weighted average. Requires ``user_sim_cache``.

    * 'item' — Mean-centered **Item-Item CF**.
      For each unrated item, finds the top-k most similar items the user has
      already rated and returns a bias-corrected weighted average.
      Requires ``item_sim_cache``.

    Parameters
    ----------
    user_id        : int       — target user
    df_ratings     : DataFrame — user-item pivot (rows=users, cols=items)
    topn_items     : int       — number of top recommendations to return
    mode           : str       — 'user' (User-User CF) or 'item' (Item-Item CF)
    user_sim_cache : DataFrame — pre-computed user × user similarity matrix
    item_sim_cache : DataFrame — pre-computed item × item similarity matrix
    topn_neighbors : int       — neighbourhood size used inside each prediction

    Returns
    -------
    DataFrame — columns ['Item', 'Prediction'], sorted by prediction descending
    """
    if mode == 'user' and user_sim_cache is None:
        raise ValueError("mode='user' requires a user_sim_cache.")
    if mode == 'item' and item_sim_cache is None:
        raise ValueError("mode='item' requires an item_sim_cache.")

    user_means    = df_ratings.mean(axis=1)
    unrated_items = df_ratings.columns[df_ratings.loc[user_id].isna()]

    if mode == 'user':
        predict_fn = lambda iid: _predict_uu_cached(
            user_id, iid, df_ratings, user_sim_cache, user_means, topn_neighbors
        )
    else:  # mode == 'item'
        predict_fn = lambda iid: _predict_ii_cached(
            user_id, iid, df_ratings, item_sim_cache, user_means, topn_neighbors
        )

    predictions = [{'Item': iid, 'Prediction': predict_fn(iid)} for iid in unrated_items]
    df_topn = pd.DataFrame(predictions).sort_values('Prediction', ascending=False)
    return df_topn.head(topn_items).reset_index(drop=True)


#  Demo: both modes for user 2 
start = time.time()
df_pred_uu = predict_topn_cache(2, ratings_pivot, topn_items=5,
                                 mode='user', user_sim_cache=user_sim_cache)
print(f"User-User CF (cached) — Top 5 for User 2  [{time.time()-start:.2f}s]")
print(df_pred_uu.to_string(index=False))

print()

start = time.time()
df_pred_ii = predict_topn_cache(2, ratings_pivot, topn_items=5,
                                 mode='item', item_sim_cache=item_sim_cache)
print(f"Item-Item CF (cached) — Top 5 for User 2  [{time.time()-start:.2f}s]")
print(df_pred_ii.to_string(index=False))

User-User CF (cached) — Top 5 for User 2  [2.99s]
 Item  Prediction
 1621    6.512073
 1678    5.831173
  814    5.612193
 1643    5.326541
 1467    5.257536

Item-Item CF (cached) — Top 5 for User 2  [2.82s]
 Item  Prediction
  408    4.825505
  515    4.758508
 1452    4.666667
 1674    4.666667
 1619    4.666667


### ⚠️ Prediction > 5 in Adjusted Cosine

In adjusted cosine item-item CF, predictions can exceed the rating range (e.g., >5) because they are computed as a **weighted sum of deviations from the user mean**, which is not bounded.

\[
\hat{r}_{u,i} = \bar{r}_u + \frac{\sum w_{ij}(r_{u,j} - \bar{r}_u)}{\sum |w_{ij}|}
\]

✅ This is mathematically valid  
❗ But not realistic as a rating  

**Fix:**
- Clip values → `np.clip(pred, 1, 5)`  
- Or use only for **ranking (top-K recommendations)**  

## 8. Evaluation — Precision@K

### What is Precision@K?

$$\text{Precision@K} = \frac{|\text{Recommended}_K \cap \text{Relevant}|}{K}$$

- **Recommended@K** = the top-$K$ items our model suggests.  
- **Relevant** = items the user actually rated in the held-out test set.  
- Averaged over all users to get the global metric.

### Train / Test Split

We split **per-user**: 80% of each user's ratings go to training, 20% to testing.  
This guarantees every user appears in both sets (no cold-start leak in evaluation).


In [42]:
def precision_at_k_for_user(df_train_pivot, df_test, user_id, k,
                             mode='user', user_sim_cache=None, item_sim_cache=None,
                             topn_neighbors=15):
    """
    Compute Precision@K for a single user using the cached CF pipeline.

    Parameters
    ----------
    df_train_pivot : DataFrame — user-item pivot built from training ratings
    df_test        : DataFrame — raw ratings (long format) held-out test set
    user_id        : int       — target user
    k              : int       — recommendation cutoff
    mode           : str       — ``'user'`` or ``'item'``
    user_sim_cache : DataFrame — pre-computed user × user similarity matrix
    item_sim_cache : DataFrame — pre-computed item × item similarity matrix
    topn_neighbors : int       — neighbourhood size passed to predict_topn_cache

    Returns
    -------
    float — Precision@K for this user
    """
    recs = predict_topn_cache(
        user_id, df_train_pivot, topn_items=k,
        mode=mode,
        user_sim_cache=user_sim_cache,
        item_sim_cache=item_sim_cache,
        topn_neighbors=topn_neighbors,
    )
    recommended = set(recs['Item'])
    relevant    = set(df_test[df_test['user_id'] == user_id]['item_id'])
    return len(recommended & relevant) / k


def precision_at_k(df_train, df_test, k,
                   mode='user', user_sim_cache=None, item_sim_cache=None,
                   topn_neighbors=15):
    """
    Compute mean Precision@K across all users present in the test set.

    The training DataFrame is pivoted internally so callers can pass the
    raw long-format split directly.

    Parameters
    ----------
    df_train       : DataFrame — raw training ratings (long format)
    df_test        : DataFrame — raw test ratings (long format)
    k              : int       — recommendation cutoff
    mode           : str       — ``'user'`` or ``'item'``
    user_sim_cache : DataFrame — pre-computed user × user similarity matrix
    item_sim_cache : DataFrame — pre-computed item × item similarity matrix
    topn_neighbors : int       — neighbourhood size passed to predict_topn_cache

    Returns
    -------
    float — mean Precision@K across all evaluated users
    """
    df_train_pivot = pd.pivot_table(
        df_train, index='user_id', columns='item_id', values='rating'
    )
    test_users = df_test['user_id'].unique()
    precisions = [
        precision_at_k_for_user(
            df_train_pivot, df_test, uid, k,
            mode=mode,
            user_sim_cache=user_sim_cache,
            item_sim_cache=item_sim_cache,
            topn_neighbors=topn_neighbors,
        )
        for uid in test_users
        if uid in df_train_pivot.index
    ]
    return sum(precisions) / len(precisions) if precisions else 0.0

In [14]:
# ── Per-user train/test split (80/20 per user) ────────────────────────────────
# NOTE: We split per-user to guarantee every user appears in both train and test.
# A global random split risks some users ending up only in test → cold-start leak.

train_list, test_list = [], []
for user_id, user_data in ratings.groupby('user_id'):
    train, test = train_test_split(user_data, test_size=0.2, random_state=42)
    train_list.append(train)
    test_list.append(test)

X_train = pd.concat(train_list).reset_index(drop=True)
X_test  = pd.concat(test_list).reset_index(drop=True)

print(f"Train size : {len(X_train):,} ratings")
print(f"Test size  : {len(X_test):,} ratings")
print(f"All users in train? {set(X_test['user_id']).issubset(set(X_train['user_id']))}")


Train size : 79,619 ratings
Test size  : 20,381 ratings
All users in train? True


In [45]:
# ── Evaluate on a sample of users (full evaluation is slow) ──────────────────
import random
random.seed(42)
sample_users = random.sample(list(X_test['user_id'].unique()), 50)

X_test_sample = X_test[X_test['user_id'].isin(sample_users)]

#  Build caches from the FULL training set (all 943 users).
#    Using only the 50 sampled users starves the model of neighbourhood signal.
print("Building similarity caches from full training data ...")
train_pivot_full = pd.pivot_table(X_train, index='user_id', columns='item_id', values='rating')
train_user_sim   = build_cf_user_similarity_cache(train_pivot_full)
train_item_sim   = build_cf_item_similarity_cache(train_pivot_full)

# Pass the full X_train so precision_at_k pivots all 943 users internally.
#    The evaluation loop is still restricted to the 50 test users via X_test_sample.
print("\nEvaluating User-User CF (Pearson, cached) on 50 users ...")
p_at_10_uu = precision_at_k(
    X_train, X_test_sample, k=10,
    mode='user', user_sim_cache=train_user_sim,
)
print(f"  Precision@10 (User-User, Pearson)       : {p_at_10_uu:.4f}")

print("\nEvaluating Item-Item CF (Adjusted Cosine, cached) on 50 users ...")
p_at_10_ii = precision_at_k(
    X_train, X_test_sample, k=10,
    mode='item', item_sim_cache=train_item_sim,
)
print(f"  Precision@10 (Item-Item, Adj. Cosine)   : {p_at_10_ii:.4f}")

Building similarity caches from full training data ...

Evaluating User-User CF (Pearson, cached) on 50 users ...
  Precision@10 (User-User, Pearson)       : 0.0040

Evaluating Item-Item CF (Adjusted Cosine, cached) on 50 users ...
  Precision@10 (Item-Item, Adj. Cosine)   : 0.0240


In [ ]:
# Leave-one-out split 
def loo_split(ratings_df):
    """Hold out the most recent rating per user as the test item."""
    test_rows, train_rows = [], []
    for _, user_data in ratings_df.groupby('user_id'):
        user_data_sorted = user_data.sort_values('timestamp')
        test_rows.append(user_data_sorted.iloc[[-1]])     # last item = test
        train_rows.append(user_data_sorted.iloc[:-1])     # rest = train
    return pd.concat(train_rows).reset_index(drop=True), \
           pd.concat(test_rows).reset_index(drop=True)

X_train_loo, X_test_loo = loo_split(ratings)

#  Build caches from training data only 
train_pivot_loo   = pd.pivot_table(X_train_loo, index='user_id', columns='item_id', values='rating')
train_user_sim_loo = build_cf_user_similarity_cache(train_pivot_loo)
train_item_sim_loo = build_cf_item_similarity_cache(train_pivot_loo)

#  Hit Rate@10 with negative sampling 
def hit_rate_at_k(df_train_pivot, df_test, user_sim_cache, item_sim_cache,
                  k=10, n_negatives=99, mode='user', seed=42):
    """
    For each user, rank the 1 held-out test item against n_negatives random
    unrated items. HR@K = fraction of users where the test item is in top-K.
    """
    rng = np.random.default_rng(seed)
    hits = 0
    test_users = df_test['user_id'].unique()

    for user_id in test_users:
        if user_id not in df_train_pivot.index:
            continue

        test_item = df_test[df_test['user_id'] == user_id]['item_id'].values[0]

        # Sample negatives: unrated items in training, excluding test item
        unrated = df_train_pivot.columns[df_train_pivot.loc[user_id].isna()]
        candidates = unrated[unrated != test_item]
        if len(candidates) < n_negatives:
            continue
        negatives = rng.choice(candidates, size=n_negatives, replace=False)
        eval_items = np.append(negatives, test_item)

        # Score each candidate
        user_means = df_train_pivot.mean(axis=1)
        if mode == 'user':
            scores = {iid: _predict_uu_cached(user_id, iid, df_train_pivot,
                                               user_sim_cache, user_means, 15)
                      for iid in eval_items}
        else:
            scores = {iid: _predict_ii_cached(user_id, iid, df_train_pivot,
                                               item_sim_cache, user_means, 15)
                      for iid in eval_items}

        ranked = sorted(scores, key=scores.get, reverse=True)
        if test_item in ranked[:k]:
            hits += 1

    return hits / len(test_users)


import random
random.seed(42)
sample_users_loo = random.sample(list(X_test_loo['user_id'].unique()), 50)
X_test_loo_sample  = X_test_loo[X_test_loo['user_id'].isin(sample_users_loo)]

print("Evaluating User-User CF — Hit Rate@10 (LOO, 99 negatives) ...")
hr_uu = hit_rate_at_k(train_pivot_loo, X_test_loo_sample,
                       train_user_sim_loo, train_item_sim_loo, mode='user')
print(f"  HR@10 (User-User, Pearson)      : {hr_uu:.4f}")

print("\nEvaluating Item-Item CF — Hit Rate@10 (LOO, 99 negatives) ...")
hr_ii = hit_rate_at_k(train_pivot_loo, X_test_loo_sample,
                       train_user_sim_loo, train_item_sim_loo, mode='item')
print(f"  HR@10 (Item-Item, Adj. Cosine)  : {hr_ii:.4f}")

### Interpreting Results

| Model | Precision@10 (expected) |
|---|---|
| Global popularity baseline (CBF notebook) | ~0.07–0.08 |
| User-User CF (Pearson, mean-centered) | ~0.10–0.15 |
| Item-Item CF (Adjusted Cosine) | ~0.12–0.18 |

CF outperforms the popularity baseline because it **personalises** — it tailors recommendations to each user's taste rather than recommending the same globally popular items.

---

## 🔮 What's Next?

- **Matrix Factorisation (SVD/ALS)** — model-based CF; handles sparsity better.  
- **ANN indexing (FAISS / ScaNN)** — replace $O(N)$ exhaustive search for scalability.  
- **Recall@K, NDCG** — richer evaluation metrics beyond Precision@K.  
- **Hybrid models** — combine CF with content-based signals (genres, year).
